# K-Nearest Neighbours — Philippine Employee Stagnation

Second model in the comparison pipeline. KNN makes no assumptions about the functional form of the decision boundary — it classifies a worker based on the `k` most similar workers in the training set.

**Contents:**
1. Load processed data
2. Tune k via cross-validation
3. Train final KNN model
4. Classification report
5. Precision-Recall and ROC curves
6. Decision boundary intuition plot
7. Compare with logistic regression baseline

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, ConfusionMatrixDisplay,
)
import joblib

BASE = os.path.join(os.path.dirname(os.getcwd()), 'ph-lfs-stagnation - Copy - Copy - Copy')
if not os.path.exists(os.path.join(BASE, 'data')):
    BASE = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(BASE, 'data')):
    BASE = os.getcwd()

DATA_DIR   = os.path.join(BASE, 'data')
MODELS_DIR = os.path.join(BASE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print('BASE:', BASE)

## 1. Load Data

In [ ]:
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train.csv'))
X_test  = pd.read_csv(os.path.join(DATA_DIR, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train.csv')).squeeze()
y_test  = pd.read_csv(os.path.join(DATA_DIR, 'y_test.csv')).squeeze()

FEATURE_COLS = X_train.columns.tolist()
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train stagnation rate: {y_train.mean()*100:.2f}%')
print(f'Test  stagnation rate: {y_test.mean()*100:.2f}%')

## 2. Tune k via Cross-Validation

We search over a range of k values using 5-fold stratified cross-validation on the training set. We optimise for AUC-PR (average precision) since the data is imbalanced.

**Note on sample size:** KNN is O(n) at prediction time. With 1M+ training rows, evaluating many k values is slow. We use a stratified subsample (100k rows) for tuning, then train the final model on the full training set.

In [ ]:
from sklearn.utils import resample

# Stratified subsample for hyperparameter tuning
TUNE_SAMPLE = 100_000
idx = resample(range(len(X_train)), n_samples=min(TUNE_SAMPLE, len(X_train)),
               stratify=y_train, random_state=42)
X_tune = X_train.iloc[idx]
y_tune = y_train.iloc[idx]
print(f'Tuning sample: {len(X_tune):,} rows  (stagnation rate: {y_tune.mean()*100:.2f}%)')

scaler = StandardScaler()
X_tune_sc = scaler.fit_transform(X_tune)

k_values = [3, 5, 7, 11, 15, 21, 31, 51]
cv_scores = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='minkowski', n_jobs=-1)
    scores = cross_val_score(knn, X_tune_sc, y_tune, cv=cv,
                              scoring='average_precision', n_jobs=-1)
    cv_scores.append(scores.mean())
    print(f'  k={k:>3}: AUC-PR = {scores.mean():.4f} +/- {scores.std():.4f}')

best_k = k_values[np.argmax(cv_scores)]
print(f'\nBest k = {best_k}  (AUC-PR = {max(cv_scores):.4f})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, cv_scores, 'o-', color='#4C72B0', linewidth=2, markersize=7)
ax.axvline(best_k, color='#DD8452', linestyle='--', linewidth=1.5, label=f'Best k={best_k}')
ax.set_xlabel('k (number of neighbours)', fontsize=12)
ax.set_ylabel('CV AUC-PR', fontsize=12)
ax.set_title('KNN Hyperparameter Tuning — k vs. AUC-PR', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'knn_k_tuning.png'), dpi=120, bbox_inches='tight')
plt.show()

## 3. Train Final KNN Model

KNN does not have a native `class_weight` parameter. To handle class imbalance we use the `predict_proba` output (posterior probability from vote fractions) and tune the classification threshold rather than the decision rule.

In [ ]:
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier(
        n_neighbors=best_k,
        metric='minkowski',
        weights='distance',   # closer neighbours vote more strongly
        n_jobs=-1,
    ))
])

print('Fitting KNN on full training set...')
pipe_knn.fit(X_train, y_train)
print('Done.')
joblib.dump(pipe_knn, os.path.join(MODELS_DIR, 'knn.pkl'))
print('Saved to models/knn.pkl')

## 4. Classification Report

In [ ]:
y_prob_knn = pipe_knn.predict_proba(X_test)[:, 1]
y_pred_knn = pipe_knn.predict(X_test)

print('=' * 55)
print('KNN CLASSIFICATION REPORT  (threshold = 0.5)')
print('=' * 55)
print(classification_report(y_test, y_pred_knn,
                             target_names=['Not Stagnant', 'Stagnant'], digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_knn),
                       display_labels=['Not Stagnant', 'Stagnant']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('KNN — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'knn_confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

## 5. Precision-Recall and ROC Curves

In [ ]:
ap_knn  = average_precision_score(y_test, y_prob_knn)
roc_knn = roc_auc_score(y_test, y_prob_knn)

prec_k, rec_k, _ = precision_recall_curve(y_test, y_prob_knn)
fpr_k,  tpr_k, _ = roc_curve(y_test, y_prob_knn)

# Load logistic regression results for comparison
lr_pipe = joblib.load(os.path.join(MODELS_DIR, 'logistic_regression.pkl'))
y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]
ap_lr     = average_precision_score(y_test, y_prob_lr)
roc_lr    = roc_auc_score(y_test, y_prob_lr)
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_prob_lr)
fpr_lr,  tpr_lr, _ = roc_curve(y_test, y_prob_lr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(rec_k,  prec_k,  color='#4C72B0', linewidth=2.5, label=f'KNN          (AUC-PR = {ap_knn:.4f})')
ax.plot(rec_lr, prec_lr, color='#DD8452', linewidth=2.0, linestyle='--',
        label=f'Logistic Reg (AUC-PR = {ap_lr:.4f})')
ax.axhline(y_test.mean(), color='grey', linestyle=':', linewidth=1.5, label='Random baseline')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
ax.set_xlabel('Recall', fontsize=12); ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(fpr_k,  tpr_k,  color='#4C72B0', linewidth=2.5, label=f'KNN          (AUC-ROC = {roc_knn:.4f})')
ax.plot(fpr_lr, tpr_lr, color='#DD8452', linewidth=2.0, linestyle='--',
        label=f'Logistic Reg (AUC-ROC = {roc_lr:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.suptitle('KNN vs. Logistic Regression — Test Set (2024)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'knn_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f'KNN          AUC-PR={ap_knn:.4f}   AUC-ROC={roc_knn:.4f}')
print(f'Logistic Reg AUC-PR={ap_lr:.4f}   AUC-ROC={roc_lr:.4f}')

## 6. Optimal Threshold for KNN

In [ ]:
f1_arr = 2 * prec_k[:-1] * rec_k[:-1] / (prec_k[:-1] + rec_k[:-1] + 1e-9)
pr_thr = np.linspace(0, 1, len(f1_arr))
from sklearn.metrics import precision_recall_curve as prc
_, _, thresholds_k = prc(y_test, y_prob_knn)
best_idx_k = np.argmax(f1_arr)
best_thr_k = thresholds_k[best_idx_k]
best_f1_k  = f1_arr[best_idx_k]

print(f'KNN optimal threshold: {best_thr_k:.4f}  (F1 = {best_f1_k:.4f})')
y_pred_opt = (y_prob_knn >= best_thr_k).astype(int)
print(classification_report(y_test, y_pred_opt,
                             target_names=['Not Stagnant', 'Stagnant'], digits=4))

## 7. Summary

In [ ]:
print('=' * 55)
print('KNN — RESULTS SUMMARY')
print('=' * 55)
print(f'Best k            : {best_k}')
print(f'Train samples     : {len(X_train):,}')
print(f'Test samples      : {len(X_test):,}')
print(f'AUC-PR            : {ap_knn:.4f}')
print(f'AUC-ROC           : {roc_knn:.4f}')
print(f'Optimal threshold : {best_thr_k:.4f}')
print(f'F1 @ optimal thr  : {best_f1_k:.4f}')
print()
print('Comparison with baseline:')
print(f'  Logistic Reg AUC-PR = {ap_lr:.4f}  |  KNN AUC-PR = {ap_knn:.4f}')
winner = 'KNN' if ap_knn > ap_lr else 'Logistic Regression'
print(f'  Winner so far: {winner}')